In [2]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

# Load matches in chronological order
matches = pd.read_csv(
    "data/processed/epl_matches_clean.csv",
    parse_dates=["Date"]
).sort_values("Date").reset_index(drop=True)

def recent_team_stats(history, last_n=5):
    """Return a team's statistics from its last N matches."""
    recent = history[-last_n:]

    if len(recent) == 0:
        return {
            "matches_before": 0,
            "form_points_5": np.nan,
            "avg_goals_for_5": np.nan,
            "avg_goals_against_5": np.nan,
        }

    return {
        "matches_before": len(history),
        "form_points_5": sum(game["points"] for game in recent),
        "avg_goals_for_5": np.mean([game["goals_for"] for game in recent]),
        "avg_goals_against_5": np.mean([game["goals_against"] for game in recent]),
    }

team_history = defaultdict(list)
feature_rows = []

# Go through matches from oldest to newest
for _, match in matches.iterrows():
    home_team = match["HomeTeam"]
    away_team = match["AwayTeam"]

    # Read each team's history BEFORE this match is added
    home_stats = recent_team_stats(team_history[home_team])
    away_stats = recent_team_stats(team_history[away_team])

    feature_rows.append({
        **match.to_dict(),
        **{f"home_{name}": value for name, value in home_stats.items()},
        **{f"away_{name}": value for name, value in away_stats.items()},
    })

    # Only now add this completed match to each team's history
    home_goals = match["FTHG"]
    away_goals = match["FTAG"]

    if home_goals > away_goals:
        home_points, away_points = 3, 0
    elif home_goals < away_goals:
        home_points, away_points = 0, 3
    else:
        home_points, away_points = 1, 1

    team_history[home_team].append({
        "goals_for": home_goals,
        "goals_against": away_goals,
        "points": home_points,
    })

    team_history[away_team].append({
        "goals_for": away_goals,
        "goals_against": home_goals,
        "points": away_points,
    })

feature_data = pd.DataFrame(feature_rows)

# Keep matches only after both teams have at least five previous matches
model_data = feature_data[
    (feature_data["home_matches_before"] >= 5) &
    (feature_data["away_matches_before"] >= 5)
].copy()

# Save the model-ready feature dataset
output_file = Path("data/processed/epl_features.csv")
model_data.to_csv(output_file, index=False)

print(f"All matches: {len(feature_data)}")
print(f"Matches ready for modelling: {len(model_data)}")
print("\nFeature columns created:")
print([
    "home_form_points_5",
    "away_form_points_5",
    "home_avg_goals_for_5",
    "away_avg_goals_for_5",
    "home_avg_goals_against_5",
    "away_avg_goals_against_5",
])

display(model_data.head())

All matches: 4230
Matches ready for modelling: 4106

Feature columns created:
['home_form_points_5', 'away_form_points_5', 'home_avg_goals_for_5', 'away_avg_goals_for_5', 'home_avg_goals_against_5', 'away_avg_goals_against_5']


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,home_matches_before,home_form_points_5,home_avg_goals_for_5,home_avg_goals_against_5,away_matches_before,away_form_points_5,away_avg_goals_for_5,away_avg_goals_against_5
50,2015-16,2015-09-19,Aston Villa,West Brom,0,1,A,5,4.0,1.2,1.6,5,5.0,0.6,1.2
51,2015-16,2015-09-19,Bournemouth,Sunderland,2,0,H,5,4.0,1.2,1.8,5,2.0,1.2,2.2
52,2015-16,2015-09-19,Chelsea,Arsenal,2,0,H,5,4.0,1.4,2.4,5,10.0,1.0,0.6
53,2015-16,2015-09-19,Man City,West Ham,1,2,A,5,15.0,2.2,0.0,5,9.0,2.2,1.2
54,2015-16,2015-09-19,Newcastle,Watford,1,2,A,5,2.0,0.4,1.4,5,6.0,0.6,0.8
